In [1]:
import pandas as pd
import json


In [2]:
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

In [3]:
print(ledger.head())
print(gateway.head())

  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       850.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R004       2026-03-02        M003      2100.0  success   
4           R005       2026-03-03        M004      7200.0  success   

  payment_method  
0            UPI  
1           Card  
2         Wallet  
3           Card  
4           Card  
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       900.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R005       2026-03-03        M004      7200.0   failed   
4           R006       2026-03-03        M002       950.0  success   

  payment_method  
0            UPI  
1     

In [4]:
print(ledger.columns)
print(gateway.columns)

Index(['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd',
       'status', 'payment_method'],
      dtype='object')
Index(['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd',
       'status', 'payment_method'],
      dtype='object')


In [5]:
print("Ledger duplicates:", ledger.duplicated().sum())
print("Gateway duplicates:", gateway.duplicated().sum())

Ledger duplicates: 0
Gateway duplicates: 0


In [6]:
print("\nLedger Null Values:")
print(ledger.isnull().sum())

print("\nGateway Null Values:")
print(gateway.isnull().sum())


Ledger Null Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Gateway Null Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [7]:
missing_in_gateway = ledger[
    ~ledger['transaction_id'].isin(gateway['transaction_id'])
]

print(missing_in_gateway.head())

  transaction_id transaction_date merchant_id  amount_usd   status  \
3           R004       2026-03-02        M003      2100.0  success   
9           R010       2026-03-05        M004      2500.0  success   

  payment_method  
3           Card  
9         Wallet  


In [8]:
missing_in_ledger = gateway[
    ~gateway['transaction_id'].isin(ledger['transaction_id'])
]

print(missing_in_ledger.head())

  transaction_id transaction_date merchant_id  amount_usd   status  \
8           R011       2026-03-05        M003      1800.0  success   

  payment_method  
8           Card  


In [9]:
merged = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)

amount_mismatches = merged[
    merged['amount_usd_ledger'] != merged['amount_usd_gateway']
]

print(amount_mismatches.head())

  transaction_id transaction_date_ledger merchant_id_ledger  \
1           R002              2026-03-01               M002   
6           R008              2026-03-04               M001   

   amount_usd_ledger status_ledger payment_method_ledger  \
1              850.0       success                  Card   
6              640.0       success                  Card   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
1               2026-03-01                M002               900.0   
6               2026-03-04                M001               600.0   

  status_gateway payment_method_gateway  
1        success                   Card  
6        success                   Card  


In [10]:
status_mismatches = merged[
    merged['status_ledger'] != merged['status_gateway']
]

print(status_mismatches.head())

  transaction_id transaction_date_ledger merchant_id_ledger  \
3           R005              2026-03-03               M004   

   amount_usd_ledger status_ledger payment_method_ledger  \
3             7200.0       success                  Card   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
3               2026-03-03                M004              7200.0   

  status_gateway payment_method_gateway  
3         failed                   Card  


In [36]:
amount_mismatches.to_csv("amount_mismatches.csv", index=False)

status_mismatches.to_csv("status_mismatches.csv", index=False)

In [14]:
missing_in_gateway = missing_in_gateway.copy()
missing_in_ledger = missing_in_ledger.copy()
amount_mismatches = amount_mismatches.copy()
status_mismatches = status_mismatches.copy()

missing_in_gateway["issue_type"] = "missing_in_gateway"
missing_in_ledger["issue_type"] = "missing_in_ledger"
amount_mismatches["issue_type"] = "amount_mismatch"
status_mismatches["issue_type"] = "status_mismatch"

In [15]:
reconciliation_report = pd.concat([
    missing_in_gateway,
    missing_in_ledger,
    amount_mismatches,
    status_mismatches
])

reconciliation_report = reconciliation_report.drop_duplicates()

In [16]:
reconciliation_report = reconciliation_report.drop_duplicates()


In [18]:
reconciliation_report.to_csv("reconciliation_report.csv", index=False)

In [20]:
import os

os.makedirs("01_data/processed", exist_ok=True)

reconciliation_report.to_csv(
    "01_data/processed/reconciliation_report.csv",
    index=False
)

In [21]:
import os
print(os.listdir("01_data/processed"))

['reconciliation_report.csv']


In [22]:
from google.colab import files
files.download("reconciliation_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
missing_in_gateway.to_csv("missing_in_gateway.csv", index=False)
missing_in_ledger.to_csv("missing_in_ledger.csv", index=False)
amount_mismatches.to_csv("amount_mismatches.csv", index=False)
status_mismatches.to_csv("status_mismatches.csv", index=False)

In [24]:
reconciliation_report.to_csv("reconciliation_report.csv", index=False)

In [26]:
import json

summary_metrics = {
    "total_ledger_rows": len(ledger),
    "total_gateway_rows": len(gateway),
    "missing_in_gateway_count": len(missing_in_gateway),
    "missing_in_ledger_count": len(missing_in_ledger),
    "amount_mismatch_count": len(amount_mismatches),
    "status_mismatch_count": len(status_mismatches),
    "reconciliation_issue_count": len(reconciliation_report),

    # SAFE FIX ↓
    "ledger_total_amount": ledger.select_dtypes(include='number').sum().sum(),
    "gateway_total_amount": gateway.select_dtypes(include='number').sum().sum(),
    "amount_at_risk": amount_mismatches.select_dtypes(include='number').sum().sum()
}

with open("summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

In [27]:
import os
print(os.listdir())

['.config', 'amount_mismatches.csv', '01_data', 'reconciliation_report.csv', 'summary_metrics.json', 'missing_in_gateway.csv', 'status_mismatches.csv', 'gateway.csv', 'missing_in_ledger.csv', 'ledger.csv', 'api_response_sample.json', 'sample_data']


In [28]:
import os

os.makedirs("01_data/processed", exist_ok=True)

In [29]:
missing_in_gateway.to_csv("01_data/processed/missing_in_gateway.csv", index=False)

missing_in_ledger.to_csv("01_data/processed/missing_in_ledger.csv", index=False)

amount_mismatches.to_csv("01_data/processed/amount_mismatches.csv", index=False)

status_mismatches.to_csv("01_data/processed/status_mismatches.csv", index=False)

reconciliation_report.to_csv("01_data/processed/reconciliation_report.csv", index=False)

In [12]:
import os

os.makedirs("04_python", exist_ok=True)
os.makedirs("01_data/processed", exist_ok=True)

In [32]:
import json

with open("04_python/summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

In [15]:
print(os.listdir("04_python"))

[]


In [14]:
print(os.listdir("01_data/processed"))
print(os.listdir("04_python"))

[]
[]


In [4]:
import json
import pandas as pd

with open("api_response_sample.json", "r") as f:
    data = json.load(f)

In [5]:
print(type(data))
print(data)

<class 'dict'>
{'generated_at': '2026-03-07T10:00:00Z', 'source': 'QuickPay Settlement API', 'batches': [{'batch_id': 'B001', 'merchant': {'merchant_id': 'M001', 'merchant_name': 'Alpha Mart', 'region': 'APAC'}, 'settlements': [{'settlement_id': 'S001', 'amount_usd': 1520.5, 'status': 'settled', 'processed_at': '2026-03-07T08:10:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S002', 'amount_usd': 980.0, 'status': 'pending', 'processed_at': '2026-03-07T08:45:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S003', 'amount_usd': 640.0, 'status': 'settled', 'processed_at': '2026-03-07T09:15:00Z', 'bank': {'name': 'Bank B', 'country': 'SG'}}]}, {'batch_id': 'B002', 'merchant': {'merchant_id': 'M004', 'merchant_name': 'Delta Travels', 'region': 'US'}, 'settlements': [{'settlement_id': 'S004', 'amount_usd': 2100.0, 'status': 'settled', 'processed_at': '2026-03-07T08:20:00Z', 'bank': {'name': 'Bank C', 'country': 'US'}}, {'settlement_id': 'S005', 'a

In [6]:
df = pd.json_normalize(data)

In [7]:
df = pd.json_normalize(data, sep="_")

In [8]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [16]:
df.to_csv("01_data/processed/api_normalized.csv", index=False)

In [17]:
import os

print(os.listdir("01_data/processed"))

['api_normalized.csv']


In [19]:
from google.colab import files

files.download("01_data/processed/api_normalized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import pandas as pd

ledger = pd.read_csv("ledger.csv")
gateway = pd.read_csv("gateway.csv")

print(ledger.head())
print(gateway.head())

  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       850.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R004       2026-03-02        M003      2100.0  success   
4           R005       2026-03-03        M004      7200.0  success   

  payment_method  
0            UPI  
1           Card  
2         Wallet  
3           Card  
4           Card  
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       900.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R005       2026-03-03        M004      7200.0   failed   
4           R006       2026-03-03        M002       950.0  success   

  payment_method  
0            UPI  
1     

In [26]:
cleaned_transactions = pd.merge(
    ledger,
    gateway,
    on="transaction_id",
    how="inner",
    suffixes=("_ledger", "_gateway")
)

print(cleaned_transactions.head())

  transaction_id transaction_date_ledger merchant_id_ledger  \
0           R001              2026-03-01               M001   
1           R002              2026-03-01               M002   
2           R003              2026-03-02               M001   
3           R005              2026-03-03               M004   
4           R006              2026-03-03               M002   

   amount_usd_ledger status_ledger payment_method_ledger  \
0             1200.0       success                   UPI   
1              850.0       success                  Card   
2              500.0       success                Wallet   
3             7200.0       success                  Card   
4              950.0       success                   UPI   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
0               2026-03-01                M001              1200.0   
1               2026-03-01                M002               900.0   
2               2026-03-02                M001    

In [27]:
daily_summary = cleaned_transactions.groupby("transaction_id").size().reset_index(name="count")
print(daily_summary.head())

  transaction_id  count
0           R001      1
1           R002      1
2           R003      1
3           R005      1
4           R006      1


In [28]:
print(cleaned_transactions.columns)
print(cleaned_transactions.head())

Index(['transaction_id', 'transaction_date_ledger', 'merchant_id_ledger',
       'amount_usd_ledger', 'status_ledger', 'payment_method_ledger',
       'transaction_date_gateway', 'merchant_id_gateway', 'amount_usd_gateway',
       'status_gateway', 'payment_method_gateway'],
      dtype='object')
  transaction_id transaction_date_ledger merchant_id_ledger  \
0           R001              2026-03-01               M001   
1           R002              2026-03-01               M002   
2           R003              2026-03-02               M001   
3           R005              2026-03-03               M004   
4           R006              2026-03-03               M002   

   amount_usd_ledger status_ledger payment_method_ledger  \
0             1200.0       success                   UPI   
1              850.0       success                  Card   
2              500.0       success                Wallet   
3             7200.0       success                  Card   
4              950.0   

In [29]:
daily_summary = cleaned_transactions.groupby("transaction_id").size().reset_index(name="count")
daily_summary.to_csv("01_data/processed/daily_summary.csv", index=False)

In [30]:
if "payment_method" in cleaned_transactions.columns:
    payment_method_breakdown = cleaned_transactions.groupby("payment_method").size().reset_index(name="count")
else:
    payment_method_breakdown = cleaned_transactions.copy()
    payment_method_breakdown["payment_method"] = "unknown"
    payment_method_breakdown = payment_method_breakdown.groupby("payment_method").size().reset_index(name="count")

payment_method_breakdown.to_csv("01_data/processed/payment_method_breakdown.csv", index=False)

In [31]:
if "region" in cleaned_transactions.columns:
    region_breakdown = cleaned_transactions.groupby("region").size().reset_index(name="count")
else:
    region_breakdown = cleaned_transactions.copy()
    region_breakdown["region"] = "unknown"
    region_breakdown = region_breakdown.groupby("region").size().reset_index(name="count")

region_breakdown.to_csv("01_data/processed/region_breakdown.csv", index=False)

In [32]:
if "merchant_id" in cleaned_transactions.columns:
    merchant_performance_summary = cleaned_transactions.groupby("merchant_id").size().reset_index(name="count")
else:
    merchant_performance_summary = cleaned_transactions.copy()
    merchant_performance_summary["merchant_id"] = "unknown"
    merchant_performance_summary = merchant_performance_summary.groupby("merchant_id").size().reset_index(name="count")

merchant_performance_summary.to_csv("01_data/processed/merchant_performance_summary.csv", index=False)

In [33]:
import os
print(os.listdir("01_data/processed"))

['merchant_performance_summary.csv', 'region_breakdown.csv', 'api_normalized.csv', 'daily_summary.csv', 'payment_method_breakdown.csv']


In [37]:
from google.colab import files

files.download("01_data/processed/daily_summary.csv")
files.download("01_data/processed/payment_method_breakdown.csv")
files.download("01_data/processed/region_breakdown.csv")
files.download("01_data/processed/merchant_performance_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>